# Data Cleaning and Preparation

**CPSC 322 Final Project, Fall 2026** · Felix Cohen, Connor Whyte

This notebook turns the raw FPL Core Insights data (snapshot in `Inputs/data/`, from
[olbauday/FPL-Core-Insights](https://github.com/olbauday/FPL-Core-Insights)) into clean tables ready for EDA.
It supports our two classification tasks:

1. **Position classifier.** Predict whether an outfield player is a defender, midfielder or forward from his stats,
   then look at the players the model "gets wrong" (players who play like a different position than FPL lists them as).
2. **Home-win classifier.** Predict whether the home team wins, using only information available before kickoff.

All the cleaning code lives in `data_prep.py`, with comments explaining every decision. This notebook runs each step in order,
shows what it produced, and runs the checks that back up our decisions. Running every cell rebuilds the files in `clean_data/`:

| File | One row per | Used for |
|---|---|---|
| `teams.csv` | team per season | looking up team names |
| `matches.csv` | finished Premier League match | building the home-win features |
| `player_matches.csv` | player per match he actually played | building the player-season table |
| `player_seasons.csv` | outfield player per season (450+ minutes) | **position classifier** |
| `match_features.csv` | match, using pre-match information only | **home-win classifier** |

We use our own `MyPyTable` class (copied from PA5) and helper functions from PA3/PA5 (in `mysklearn/myutils.py`) instead of pandas.

In [1]:
import importlib
import data_prep
from mysklearn.mypytable import MyPyTable
from mysklearn import myutils

importlib.reload(data_prep)  # picks up edits to data_prep.py without restarting the kernel


def show(table, columns=None, num_rows=5):
    """Prints the first few rows of a MyPyTable, optionally only some columns."""
    columns = columns or table.column_names
    indexes = [table.get_column_index(name) for name in columns]
    MyPyTable(columns, [[row[i] for i in indexes] for row in table.data[:num_rows]]).pretty_print()

## Step 1: Teams

Each season has a `teams.csv` with a code and name for every club. We keep only the code and names.

The raw file also has "strength" and "elo" columns, but those are a snapshot from the day the data was downloaded,
so they already reflect how the season went. Using them as features would leak future information, so we leave them out.

In [2]:
teams = data_prep.build_teams()
print("Shape:", teams.get_shape())
print("Teams per season:", myutils.get_frequencies(teams.get_column("season")))
show(teams)

Shape: (60, 4)
Teams per season: (['2024-2025', '2025-2026', '2026-2027'], [20, 20, 20])
season       team_code  team_name    short_name
---------  -----------  -----------  ------------
2024-2025            3  Arsenal      ARS
2024-2025            7  Aston Villa  AVL
2024-2025           91  Bournemouth  BOU
2024-2025           94  Brentford    BRE
2024-2025           36  Brighton     BHA


## Step 2: Matches

What we do (details in `build_matches`):

* Load only the **Premier League** folders. The "By Gameweek" folders also hold cup and European games, which would mix competitions.
* Keep only **finished** matches. 2026-27 also lists future fixtures.
* Remove duplicate `match_id`s with `MyPyTable.find_duplicates`, in case a postponed match appears in two gameweek folders.
* Convert team codes from `36.0` to `36`, add team names, and put kickoff times into one format.
  The raw files mix three date styles.
* Add the labels: `result` (H/D/A) and `home_win` (yes/no).

In [3]:
matches = data_prep.build_matches(teams)
print("Shape:", matches.get_shape())
print("Matches per season:", myutils.get_frequencies(matches.get_column("season")))
print("Duplicate match_ids left:", len(matches.find_duplicates(["match_id"])))
print("Result (H/D/A):", myutils.get_frequencies(matches.get_column("result")))
print("Home win:", myutils.get_frequencies(matches.get_column("home_win")))
show(matches, ["season", "gameweek", "kickoff", "home_name", "away_name", "home_goals", "away_goals", "result"])

Shape: (870, 64)
Matches per season: (['2024-2025', '2025-2026', '2026-2027'], [380, 380, 110])
Duplicate match_ids left: 0
Result (H/D/A): (['A', 'D', 'H'], [277, 229, 364])
Home win: (['no', 'yes'], [506, 364])
season       gameweek  kickoff           home_name    away_name      home_goals    away_goals  result
---------  ----------  ----------------  -----------  -----------  ------------  ------------  --------
2024-2025           1  2024-08-16 19:00  Man Utd      Fulham                  1             0  H
2024-2025           1  2024-08-17 11:30  Ipswich      Liverpool               0             2  A
2024-2025           1  2024-08-17 14:00  Arsenal      Wolves                  2             0  H
2024-2025           1  2024-08-17 14:00  Everton      Brighton                0             3  A
2024-2025           1  2024-08-17 14:00  Newcastle    Southampton             1             0  H


### Check: are the ClubElo ratings recorded before or after each match?

Team strength ratings (Elo) are one of the best predictors of who wins, but only if they're recorded **before** the match.
A rating recorded after the match already "knows" the result.

The test: line up each team's matches in date order. If ratings are pre-match, a match's result shows up in the rating change
from that row **to the next** row. If they're post-match, it shows up in the change **from the previous** row. We correlate
each kind of change with the match's goal difference.

In [4]:
corr_next, corr_prev = data_prep.check_clubelo_timing(matches)
print(f"Goal difference vs rating change to the NEXT match:      r = {corr_next:.3f}")
print(f"Goal difference vs rating change from the PREVIOUS match: r = {corr_prev:.3f}")

Goal difference vs rating change to the NEXT match:      r = 0.590
Goal difference vs rating change from the PREVIOUS match: r = 0.018


The result shows up in the change to the next row (r ≈ 0.59) and not in the change from the previous row (r ≈ 0.02),
so **ClubElo ratings are pre-match** and safe to use.

### Check: how complete are the ClubElo ratings?

In [5]:
season_col = matches.get_column("season")
clubelo_col = matches.get_column("clubelo_home")
for season in data_prep.SEASONS:
    in_season = [elo for s, elo in zip(season_col, clubelo_col) if s == season]
    print(f"{season}: ClubElo missing for {in_season.count('NA')} of {len(in_season)} matches")

2024-2025: ClubElo missing for 9 of 380 matches
2025-2026: ClubElo missing for 0 of 380 matches
2026-2027: ClubElo missing for 93 of 110 matches


ClubElo is missing for most of 2026-27 (and for 2024-25 gameweek 38). So we **compute our own Elo ratings** from match results
(`compute_elo_ratings`). Each team starts at its first ClubElo value, so our ratings use the same scale. Each match then moves
both teams' ratings according to how surprising the result was. We save every rating *before* updating it,
so a match's result never affects its own row.

To check our ratings are sensible, we compare them with ClubElo wherever ClubElo exists:

In [6]:
r, num_pairs = data_prep.compare_elo_to_clubelo(matches)
print(f"Correlation between our Elo and ClubElo: r = {r:.3f} (over {num_pairs} team-match ratings)")
show(matches, ["season", "gameweek", "home_name", "away_name", "clubelo_home", "elo_home", "clubelo_away", "elo_away"])

Correlation between our Elo and ClubElo: r = 0.969 (over 1537 team-match ratings)
season       gameweek  home_name    away_name      clubelo_home    elo_home    clubelo_away    elo_away
---------  ----------  -----------  -----------  --------------  ----------  --------------  ----------
2024-2025           1  Man Utd      Fulham              1779.04      1779           1716.28      1716.3
2024-2025           1  Ipswich      Liverpool           1568.33      1568.3         1900.69      1900.7
2024-2025           1  Arsenal      Wolves              1946.9       1946.9         1677.86      1677.9
2024-2025           1  Everton      Brighton            1706.85      1706.8         1713.16      1713.2
2024-2025           1  Newcastle    Southampton         1801.8       1801.8         1599.6       1599.6


Our ratings track ClubElo very closely (r ≈ 0.97) and exist for every match. The two differ slightly because ClubElo also counts
cup and European games and ours only counts Premier League games. We keep both columns, and `match_features.csv` uses ours.

## Step 3: Player matches

The raw `playermatchstats.csv` files have one row per player per match. What we do (details in `build_player_matches`):

1. Keep only rows for matches in our cleaned matches table.
2. **Drop 0-minute rows.** These are unused substitutes who didn't play, so their stats would only drag averages toward zero.
3. **Find each player's team.** The stats file has no team column. We use `lineups.csv` (2025-26), then the per-gameweek `players.csv`
   snapshot (2025-26 and 2026-27; it shows each player's club *at that gameweek*, so January transfers are handled).
   2024-25 only has an end-of-season player list. For players who moved mid-season, `resolve_unmatched_teams` works out the
   old club: it's the one club that appears in all of his early matches.
4. **Use `player_code` as the player's identity**, not `player_id` (see the check below).
5. **Drop misleading columns:** the `_percent` columns (0% from 0 attempts looks the same as 0% from 10 attempts) and
   `start_min`/`finish_min`, which the data source says are unreliable.
6. **Fix blanks that really mean zero** in 2026-27 (see the check below).

### Check: does `player_id` stay the same across seasons?

In [7]:
import os
players_2425 = data_prep.load_csv(os.path.join(data_prep.RAW_DATA_DIR, "2024-2025", "players", "players.csv"))
players_2526 = data_prep.load_csv(os.path.join(data_prep.RAW_DATA_DIR, "2025-2026", "players.csv"))
ids_2526 = dict(zip(players_2526.get_column("player_code"), players_2526.get_column("player_id")))
both = [(code, pid) for code, pid in zip(players_2425.get_column("player_code"), players_2425.get_column("player_id"))
        if code in ids_2526]
same_id = sum(1 for code, pid in both if ids_2526[code] == pid)
print(f"Players in both seasons (matched by player_code): {len(both)}")
print(f"...of whom kept the same player_id: {same_id}")

Players in both seasons (matched by player_code): 534
...of whom kept the same player_id: 1


FPL renumbers `player_id` every season, so it can't link a player's seasons together. `player_code` can.
This matters later: when we split data into training and test sets, all of a player's seasons must go on the same side.
Otherwise the model could recognize the player instead of learning the position.

### Check: do blanks in 2026-27 mean "zero"?

In 2026-27 many stat cells are blank for players who did play. If blanks meant "unknown", `shots_on_target` should sometimes be blank for
players who *did* shoot. We count, using the table before any blanks are filled in:

In [8]:
unfilled = data_prep.build_player_matches(matches, fill_zeros=False)
print(data_prep.check_blank_zero_evidence(unfilled))

{'total_shots blank': 1536, 'total_shots above 0': 0, 'explicit 0 in total_shots': 0}


`shots_on_target` is blank only when `total_shots` is also blank, and `total_shots` never contains an explicit 0. So in 2026-27
the source writes blanks instead of zeros, and `fill_blank_zeros` turns them into 0 for outfield players. We do **not** fill:

* columns that are blank for the *entire* season (e.g. `tackles` and `defensive_contributions` in 2026-27). Those weren't recorded at all.
* goalkeepers. In 2026-27 their passing is stored in the `gk_` columns, so a blank in the outfield columns isn't a zero.
* earlier seasons, where blanks are rare and really do mean missing.

In [9]:
player_matches = data_prep.build_player_matches(matches)
print("Shape:", player_matches.get_shape())
print("Rows per season:", myutils.get_frequencies(player_matches.get_column("season")))
print("FPL position:", myutils.get_frequencies(player_matches.get_column("fpl_position")))
print("Rows with no team/opponent:", player_matches.get_column("is_home").count("NA"))
show(player_matches, ["season", "gameweek", "player_name", "team_code", "opponent_code", "is_home",
                      "fpl_position", "lineup_position", "minutes", "goals", "xg", "tackles_won"])

Shape: (26443, 64)
Rows per season: (['2024-2025', '2025-2026', '2026-2027'], [11563, 11492, 3388])
FPL position: (['DEF', 'FWD', 'GK', 'MID'], [8883, 3091, 1759, 12710])
Rows with no team/opponent: 0
season       gameweek  player_name      team_code    opponent_code  is_home    fpl_position    lineup_position      minutes    goals    xg    tackles_won
---------  ----------  -------------  -----------  ---------------  ---------  --------------  -----------------  ---------  -------  ----  -------------
2024-2025           1  Onana                    1               54  yes        GK              NA                        90        0  0                 0
2024-2025           1  Casemiro                 1               54  yes        MID             NA                        90        0  0.16              2
2024-2025           1  Mazraoui                 1               54  yes        DEF             NA                        81        0  0                 2
2024-2025           1  Mainoo

### Missing values that remain

These are left as `NA` on purpose; they are real gaps, not zeros. The main ones to know about for EDA:

In [10]:
missing = myutils.count_missing_values(player_matches)
for name, count in sorted(missing.items(), key=lambda item: -item[1]):
    print(f"{name:26s} {count:6d}  ({count / player_matches.get_shape()[0]:.0%})")

number_of_sprints           23238  (88%)
sprinting_distance          23169  (88%)
top_speed                   23055  (87%)
distance_covered            23055  (87%)
running_distance            23055  (87%)
defensive_contributions     15244  (58%)
saves_inside_box            15022  (57%)
lineup_position             14965  (57%)
started                     14965  (57%)
dispossessed                11785  (45%)
big_chances_missed           3681  (14%)
offsides                     3681  (14%)
xgot_faced                   3468  (13%)
goals_prevented              3468  (13%)
gk_accurate_long_balls       3461  (13%)
sweeper_actions              3459  (13%)
gk_accurate_passes           3459  (13%)
high_claim                   3459  (13%)
successful_dribbles          3388  (13%)
tackles                      3388  (13%)
saves                        3166  (12%)
goals_conceded               3166  (12%)
touches_opposition_box        515  (2%)
blocks                        515  (2%)
headed_clearances 

* `lineup_position` / `started` exist only for 2025-26 (no lineup files in the other seasons).
* Some stats weren't recorded in some seasons: `dispossessed` (not in 2024-25), `tackles`, `successful_dribbles`,
  `big_chances_missed`, `offsides` (not in 2026-27), `defensive_contributions` (2025-26 only).
  **For EDA:** compare positions with stats that exist in every season (e.g. `tackles_won`, `interceptions`, `clearances`, `xg`).
* Goalkeeper-only stats (`saves`, etc.) are blank for outfield players, and physical tracking stats (`distance_covered`, etc.)
  exist only for 2026-27.
* About 300 rows from late 2025-26 are missing a group of detailed stats (`blocks`, `dribbled_past`, etc.).

## Step 4: Player seasons (position classifier table)

`build_player_seasons` combines each outfield player's matches into **one row per player per season**:

* Goalkeepers and "Unknown" positions are left out (goalkeepers aren't one of our classes).
* Every stat becomes a **per-90-minute rate**, so players with different playing time can be compared fairly.
  Each rate only counts minutes from matches where that stat was recorded.
* Player-seasons under **450 minutes** (five full matches) are dropped, because rates from very little playing time are noisy.
* The label is `fpl_position` (DEF/MID/FWD). `lineup_position` (2025-26 only) is kept for the out-of-position analysis,
  **not** as a feature, since it would give the answer away.

How the minimum-minutes cutoff changes the size of the dataset:

In [11]:
for cutoff in [0, 270, 450, 900]:
    print(f"min_minutes = {cutoff:4d}: {data_prep.build_player_seasons(player_matches, cutoff).get_shape()[0]} player-seasons")

min_minutes =    0: 1447 player-seasons


min_minutes =  270: 1066 player-seasons


min_minutes =  450: 941 player-seasons
min_minutes =  900: 668 player-seasons


450 minutes keeps close to 1,000 player-seasons while removing the noisiest rows. If we want more instances, 270 minutes
(three full matches) gives just over 1,000. This is a trade-off to revisit during EDA.

In [12]:
player_seasons = data_prep.build_player_seasons(player_matches, min_minutes=450)
print("Shape:", player_seasons.get_shape())
print("Rows per season:", myutils.get_frequencies(player_seasons.get_column("season")))
print("Class label (fpl_position):", myutils.get_frequencies(player_seasons.get_column("fpl_position")))
show(player_seasons, ["season", "player_name", "fpl_position", "lineup_position", "minutes",
                      "xg_per90", "tackles_won_per90", "clearances_per90", "touches_opposition_box_per90"])

Shape: (941, 44)
Rows per season: (['2024-2025', '2025-2026', '2026-2027'], [364, 372, 205])
Class label (fpl_position): (['DEF', 'FWD', 'MID'], [374, 102, 465])
season     player_name    fpl_position    lineup_position      minutes    xg_per90    tackles_won_per90    clearances_per90    touches_opposition_box_per90
---------  -------------  --------------  -----------------  ---------  ----------  -------------------  ------------------  ------------------------------
2024-2025  A.Doucoure     MID             NA                      2577       0.145                0.978               0.664                           2.48
2024-2025  A.Fatawu       MID             NA                       577       0.064                2.34                1.872                           4.367
2024-2025  Adama          MID             NA                      1769       0.204                0.661               0.204                           4.681
2024-2025  Adams          MID             NA               

Forwards are the smallest class (about 11%), so we should use a stratified split and look at per-class results, as planned in our proposal.

Summary statistics for a few stats that exist in every season:

In [13]:
player_seasons.compute_summary_statistics(["xg_per90", "xa_per90", "touches_opposition_box_per90", "tackles_won_per90",
                                           "interceptions_per90", "clearances_per90", "aerial_duels_won_per90"]).pretty_print()

attribute                       min     max     mid        avg    median
----------------------------  -----  ------  ------  ---------  --------
xg_per90                      0       0.786  0.393   0.1439        0.088
xa_per90                      0.001   0.437  0.219   0.0906759     0.073
touches_opposition_box_per90  0      11.852  5.926   2.63664       1.934
tackles_won_per90             0.045   4.038  2.0415  1.13607       1.008
interceptions_per90           0       3.077  1.5385  0.814237      0.764
clearances_per90              0      10.706  5.353   2.32633       1.67
aerial_duels_won_per90        0       8.017  4.0085  1.40495       1.096


### First look at out-of-position players (2025-26)

Rows are FPL's label; columns are where the player was most often listed in that season's lineups:

In [14]:
fpl_col = player_seasons.get_column("fpl_position")
lineup_col = player_seasons.get_column("lineup_position")
labels = ["DEF", "MID", "FWD"]
table = []
for fpl in labels:
    table.append([fpl] + [sum(1 for f, l in zip(fpl_col, lineup_col) if f == fpl and l == lineup) for lineup in labels])
MyPyTable(["FPL label \\ lineup"] + labels, table).pretty_print()

FPL label \ lineup      DEF    MID    FWD
--------------------  -----  -----  -----
DEF                     138      8      1
MID                       2    143     36
FWD                       0      1     43


Most players line up where FPL lists them. The big exception is about 36 FPL "midfielders" who are usually listed as forwards.
These are mostly wingers, and exactly the kind of player the out-of-position analysis is about.

## Step 5: Match features (home-win classifier table)

The key rule: **every feature must be something a bettor could have known before kickoff.** So `build_match_features` never uses
a match's own stats. For each match it looks back at each team's previous **5** league matches and averages them:

* form: points, goals for/against, xG for/against, shots for/against, shots on target, big chances for/against,
  touches in the opposition box, possession
* `elo_home`, `elo_away`, `elo_diff`: our pre-match Elo ratings from Step 2
* `rest_days`: days since the team's last league match this season (`NA` for the first match of a season)
* `played_midweek`: did the team play a cup or European game in the 4 days before? (`NA` for 2024-25, which has no cup data)
* `home_history` / `away_history`: how many previous matches the averages are based on

Form carries over from the end of the previous season, but never further back. If a team has fewer than 3 previous matches
(every team at the start of 2024-25, and promoted teams), its form features are `NA` instead of an average of one or two games.

In [15]:
match_features = data_prep.build_match_features(matches)
print("Shape:", match_features.get_shape())
print("Home win label:", myutils.get_frequencies(match_features.get_column("home_win")))
print("Result:", myutils.get_frequencies(match_features.get_column("result")))
print("Home team history length:", myutils.get_frequencies(match_features.get_column("home_history")))
show(match_features, ["season", "gameweek", "home_name", "away_name", "elo_diff", "home_avg_points",
                      "away_avg_points", "home_avg_xg_for", "away_avg_xg_for", "home_win"])

Shape: (870, 43)
Home win label: (['no', 'yes'], [506, 364])
Result: (['A', 'D', 'H'], [277, 229, 364])
Home team history length: ([0, 1, 2, 3, 4, 5], [14, 12, 14, 12, 12, 806])
season       gameweek  home_name    away_name      elo_diff  home_avg_points    away_avg_points    home_avg_xg_for    away_avg_xg_for    home_win
---------  ----------  -----------  -----------  ----------  -----------------  -----------------  -----------------  -----------------  ----------
2024-2025           1  Man Utd      Fulham             62.7  NA                 NA                 NA                 NA                 yes
2024-2025           1  Ipswich      Liverpool        -332.4  NA                 NA                 NA                 NA                 no
2024-2025           1  Arsenal      Wolves            269    NA                 NA                 NA                 NA                 yes
2024-2025           1  Everton      Brighton           -6.4  NA                 NA                 NA     

### Check: no information from the match itself leaks in

We pick one match and recompute the home team's average xG and goals by hand from the matches table,
using only the five matches before that kickoff:

In [16]:
row = match_features.data[500]
match = dict(zip(match_features.column_names, row))
team = match["home_code"]
previous = []
for m_row in matches.data:
    m = dict(zip(matches.column_names, m_row))
    if m["kickoff"] < match["kickoff"] and team in (m["home_code"], m["away_code"]):
        side = "home" if m["home_code"] == team else "away"
        previous.append((m[side + "_expected_goals_xg"], m[side + "_goals"]))
last_five = previous[-5:]
print(f"Match: {match['home_name']} vs {match['away_name']}, {match['kickoff']}")
print("By hand  - avg xG for:", round(sum(x for x, _ in last_five) / 5, 3), " avg goals for:", sum(g for _, g in last_five) / 5)
print("Pipeline - avg xG for:", match["home_avg_xg_for"], " avg goals for:", match["home_avg_goals_for"])

Match: Sunderland vs Bournemouth, 2025-11-29 15:00
By hand  - avg xG for: 0.758  avg goals for: 1.4
Pipeline - avg xG for: 0.758  avg goals for: 1.4


The numbers match, so the features only use earlier matches.

### Baselines for later

The home team wins about 42% of the time, so always predicting "no" scores about 58%. A rule that predicts "yes"
whenever the home team's Elo plus a 65-point home bonus is higher than the away team's scores:

In [17]:
elo_diffs = match_features.get_column("elo_diff")
results = match_features.get_column("result")
correct = sum(1 for diff, result in zip(elo_diffs, results) if (diff + 65 > 0) == (result == "H"))
print(f"Elo rule accuracy on home_win: {correct / len(results):.3f}")
print("Missing values:", myutils.count_missing_values(match_features))

Elo rule accuracy on home_win: 0.583
Missing values: {'home_rest_days': 30, 'home_played_midweek': 380, 'home_avg_points': 40, 'home_avg_goals_for': 40, 'home_avg_goals_against': 40, 'home_avg_xg_for': 40, 'home_avg_xg_against': 40, 'home_avg_shots_for': 40, 'home_avg_shots_against': 40, 'home_avg_shots_on_target_for': 40, 'home_avg_big_chances_for': 40, 'home_avg_big_chances_against': 40, 'home_avg_box_touches_for': 40, 'home_avg_possession': 40, 'away_rest_days': 30, 'away_played_midweek': 380, 'away_avg_points': 38, 'away_avg_goals_for': 38, 'away_avg_goals_against': 38, 'away_avg_xg_for': 38, 'away_avg_xg_against': 38, 'away_avg_shots_for': 38, 'away_avg_shots_against': 38, 'away_avg_shots_on_target_for': 38, 'away_avg_big_chances_for': 38, 'away_avg_big_chances_against': 38, 'away_avg_box_touches_for': 38, 'away_avg_possession': 38}


Both baselines sit around 58%, which shows how hard this task is. Beating them, and later the bookmakers, is the goal.

## Save the cleaned tables

In [18]:
tables = {"teams.csv": teams, "matches.csv": matches, "player_matches.csv": player_matches,
          "player_seasons.csv": player_seasons, "match_features.csv": match_features}
for filename, table in tables.items():
    data_prep.save_table(table, filename)
    print(f"Saved clean_data/{filename:22s} {table.get_shape()}")

Saved clean_data/teams.csv              (60, 4)


Saved clean_data/matches.csv            (870, 64)


Saved clean_data/player_matches.csv     (26443, 64)
Saved clean_data/player_seasons.csv     (941, 44)
Saved clean_data/match_features.csv     (870, 43)


## Summary: what EDA needs to know

* **Position task:** `player_seasons.csv` has 941 rows (450+ minutes) with labels DEF / MID / FWD. Forwards are the smallest class.
  Group rows by `player_code` when splitting, so a player's seasons stay together.
  Some stats are missing for whole seasons, so prefer stats that exist every season for box plots and feature selection.
* **Home-win task:** `match_features.csv` has 870 matches, below the "thousands of instances" guideline, so worth mentioning to the instructor.
  About 40 rows per side have no form history; decide whether to drop them or classify them using Elo alone.
* **Not yet included:** bookmaker odds (needs an outside source such as football-data.co.uk) and player ratings
  (`player_match_enrichment.csv`, 2025-26 only).
* **Discretization** for our random forest and Naive Bayes, and normalization for k-NN, happen later, after EDA shows the distributions.